In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:
import sys
import subprocess
import torch

print(sys.executable)


s:\Projects\rag-dissertation\.venv\Scripts\python.exe


: 

In [2]:
#importing dataset

df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()

,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [3]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [5]:
#Creating Embaddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1709.31it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [6]:
#Vector database

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

vector_store.add_documents(texts)

InternalError: ValueError: Batch size of 22842 is greater than max batch size of 5461

In [7]:
from tqdm import tqdm

max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
BATCH = min(2000, max_bs)                            # stay safely under it

for i in tqdm(range(0, len(texts), BATCH)):
    batch = texts[i:i + BATCH]
    ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
    vector_store.add_documents(batch, ids=ids)

100%|██████████| 12/12 [1:02:40<00:00, 313.33s/it]


In [13]:
#Retrieval

retriever = vector_store.as_retriever()

ret = retriever.invoke("what is adhd?")

In [14]:
print(ret)

[]


In [ ]:
#Setting up LLM

llm = ChatOllama(
    model="gemma4:e4b", 
    )

In [ ]:
#Query LLM

response = llm.invoke("Hey how u doing today")

In [6]:
print(response.text)

I'm doing great, thank you for asking! 😊 Just here and ready to chat.

How about you? What's going on with you today?
